### Mercari PyCaret 분석기 클래스
1. TSV 데이터 로딩 지원
2. PyCaret setup, compare_models로 base model 탐색
3. 차원 축소(TF-IDF/Embedding 과 관련한 고차원 feature) 적용 가능
4. 단계별 진행 print 문구
5. tqdm 진행 표시
6. 모델 성능 지표 .json 저장
7. Submission CSV 저장
8. plot_model 시각화 저장 (../images/{model_name}_{timestamp}.png)

In [1]:
import pandas as pd
import numpy as np
import os
import json
import datetime
import gc
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

from pycaret.regression import *

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD


class MercariPyCaretAnalyzer:
    """
    Mercari Price Suggestion Challenge용 PyCaret 분석기
    - TSV/CSV 데이터 로딩
    - TF-IDF / CountVectorizer + TruncatedSVD 차원 축소
    - PyCaret setup & compare_models
    - Base model 생성, 시각화, 예측
    - Metrics JSON 저장, submission CSV 저장
    """

    def __init__(
        self, data_dir="../data", images_dir="../images", results_dir="../results"
    ):
        self.data_dir = data_dir
        self.images_dir = images_dir
        self.results_dir = results_dir

        self.train = None
        self.test = None
        self.best_model = None
        self.setup_result = None
        self.metrics = {}

        os.makedirs(self.images_dir, exist_ok=True)
        os.makedirs(self.results_dir, exist_ok=True)

    # ------------------ 데이터 로딩 ------------------
    def load_data(self, train_file="train.tsv", test_file="test.tsv", sep="\t"):
        print("📂 데이터 로딩 시작...")
        train_path = os.path.join(self.data_dir, train_file)
        test_path = os.path.join(self.data_dir, test_file)

        self.train = pd.read_csv(train_path, sep=sep)
        self.test = pd.read_csv(test_path, sep=sep)

        print(f"original data shape : train {self.train.shape}, test {self.test.shape}")

        print("✅ Price 외 결측치 처리 및 데이터 전처리 시작...")

        # price 0 제거 + NaN 제거 (train만 해당)
        self.train = self.train[self.train["price"] > 0].dropna(subset=["price"])
        print("Price NaN count:", self.train["price"].isna().sum())

        # 카테고리 대중소 분류 (train과 test 모두)
        for df_name, df in [("train", self.train), ("test", self.test)]:
            df["main_cat"], df["sub_cat"], df["sub_sub_cat"] = zip(
                *df["category_name"].apply(
                    lambda x: (
                        x.split("/")
                        if isinstance(x, str) and "/" in x
                        else ["missing", "missing", "missing"]
                    )
                )
            )

            # 결측치 처리
            df["brand_name"] = df["brand_name"].fillna("Unknown")
            df["category_name"] = df["category_name"].fillna("Unknown")
            df["item_description"] = df["item_description"].fillna("No description")

            # 인덱스 리셋
            if df_name == "train":
                self.train = df.reset_index(drop=True)
            else:
                self.test = df.reset_index(drop=True)

        # price log 변환 (train만)
        self.train["price"] = np.log1p(self.train["price"])

        print("Final Price NaN count:", self.train["price"].isna().sum())
        print("Train length:", len(self.train))
        print("\nTrain head:")
        print(self.train.head())

        print(f"\nTrain info:\n{'='*50}")
        print(self.train.info())

        print(
            f"\n✅ 데이터 로드 완료: train {self.train.shape}, test {self.test.shape}"
        )

    # ------------------ Undersampling 기능 ------------------
    def apply_undersampling(
        self,
        method="random",
        target_size=None,
        sampling_ratio=0.5,
        n_bins=10,
        random_state=23,
    ):
        """
        가격 데이터에 대한 undersampling 적용

        Parameters:
        -----------
        method : str
            'random' : 무작위 샘플링
            'stratified' : 가격 구간별 층화 샘플링
            'top_n' : 상위 N개만 선택
        target_size : int, optional
            목표 샘플 수 (None이면 sampling_ratio 사용)
        sampling_ratio : float
            샘플링 비율 (0 < ratio <= 1)
        n_bins : int
            stratified 방식 사용 시 가격 구간 수
        random_state : int
            재현성을 위한 random seed
        """
        if self.train is None:
            raise ValueError("먼저 load_data()를 실행하세요.")

        original_size = len(self.train)

        # target_size 결정
        if target_size is None:
            target_size = int(original_size * sampling_ratio)
        else:
            target_size = min(target_size, original_size)

        print(f"\n🎯 Undersampling 시작...")
        print(f"   방법: {method}")
        print(f"   원본 크기: {original_size:,}")
        print(f"   목표 크기: {target_size:,}")
        print(f"   샘플링 비율: {target_size/original_size:.2%}")

        if method == "random":
            # 무작위 샘플링
            sampled_indices = np.random.RandomState(random_state).choice(
                self.train.index, size=target_size, replace=False
            )
            self.train = self.train.loc[sampled_indices].reset_index(drop=True)

        elif method == "stratified":
            # 가격 구간별 층화 샘플링
            self.train["price_bin"] = pd.qcut(
                self.train["price"], q=n_bins, labels=False, duplicates="drop"
            )

            # 각 구간에서 동일 비율로 샘플링
            sampled = self.train.groupby("price_bin", group_keys=False).apply(
                lambda x: x.sample(
                    n=max(1, int(len(x) * sampling_ratio)), random_state=random_state
                )
            )

            self.train = sampled.drop(columns=["price_bin"]).reset_index(drop=True)

        elif method == "top_n":
            # 상위 N개만 선택 (정렬 기준은 price)
            self.train = self.train.nlargest(target_size, "price").reset_index(
                drop=True
            )

        else:
            raise ValueError(f"지원하지 않는 method: {method}")

        final_size = len(self.train)
        print(f"✅ Undersampling 완료")
        print(f"   최종 크기: {final_size:,}")
        print(f"   실제 샘플링 비율: {final_size/original_size:.2%}")
        print(f"   제거된 샘플: {original_size - final_size:,}")

        # 가격 분포 통계 출력
        print(f"\n📊 샘플링 후 가격 분포:")
        print(f"   평균: {self.train['price'].mean():.4f}")
        print(f"   중앙값: {self.train['price'].median():.4f}")
        print(f"   표준편차: {self.train['price'].std():.4f}")
        print(f"   최소: {self.train['price'].min():.4f}")
        print(f"   최대: {self.train['price'].max():.4f}")

    # ------------------ TF-IDF / CountVectorizer + 차원 축소 ------------------
    def vectorize_text(
        self,
        text_columns=["name", "item_description"],
        method="tfidf",
        max_features=50000,
        n_components=100,
    ):
        """
        text_columns: list of columns to vectorize
        method: 'tfidf' or 'count'
        max_features: Vectorizer max features
        n_components: TruncatedSVD components
        """
        print("📝 텍스트 벡터화 및 차원 축소 시작...")
        vectors = []
        feature_names = []

        for col in tqdm(text_columns, desc="Text columns"):
            print(f"▶ 컬럼: {col}")
            if method == "tfidf":
                vec = TfidfVectorizer(max_features=max_features)
            elif method == "count":
                vec = CountVectorizer(max_features=max_features)
            else:
                raise ValueError("method must be 'tfidf' or 'count'")

            combined_text = pd.concat([self.train[col], self.test[col]], axis=0)
            vec.fit(combined_text)

            train_vec = vec.transform(self.train[col])
            test_vec = vec.transform(self.test[col])

            # 차원 축소
            if n_components < train_vec.shape[1]:
                svd = TruncatedSVD(n_components=n_components, random_state=23)
                train_vec = svd.fit_transform(train_vec)
                test_vec = svd.transform(test_vec)
                print(f"   ▪ 차원 축소 완료: {train_vec.shape[1]} components")
            else:
                train_vec = train_vec.toarray()
                test_vec = test_vec.toarray()

            vectors.append((train_vec, test_vec))
            feature_names.append([f"{col}_{i}" for i in range(train_vec.shape[1])])
            # 메모리 해제
            del combined_text, vec
            gc.collect()

        # 합치기
        train_features = np.hstack([v[0] for v in vectors])
        test_features = np.hstack([v[1] for v in vectors])

        self.train_vectorized = pd.DataFrame(
            train_features, columns=[f for sub in feature_names for f in sub]
        )
        self.test_vectorized = pd.DataFrame(
            test_features, columns=[f for sub in feature_names for f in sub]
        )

        # ✅ 카테고리 변수 인코딩 및 추가
        from sklearn.preprocessing import LabelEncoder

        categorical_cols = [
            "main_cat",
            "sub_cat",
            "sub_sub_cat",
            "brand_name",
            "shipping",
        ]

        for col in categorical_cols:
            if col in self.train.columns:
                # LabelEncoder 사용
                le = LabelEncoder()

                # train + test 함께 fit
                combined = pd.concat(
                    [self.train[col].astype(str), self.test[col].astype(str)], axis=0
                )
                le.fit(combined)

                # 변환
                self.train_vectorized[col] = le.transform(
                    self.train[col].astype(str).reset_index(drop=True)
                )
                self.test_vectorized[col] = le.transform(
                    self.test[col].astype(str).reset_index(drop=True)
                )

                del combined

        print(
            f"✅ 벡터화 + 차원 축소 + 카테고리 인코딩 완료: "
            f"train {self.train_vectorized.shape}, test {self.test_vectorized.shape}"
        )

    # ------------------ PyCaret setup ------------------
    def setup_pycaret(self, session_id=23):
        if not hasattr(self, "train_vectorized"):
            raise ValueError("먼저 vectorize_text()를 실행하세요.")

        print("🔧 PyCaret setup 시작...")

        # 카테고리 컬럼이 실제로 존재하는지 확인
        categorical_cols = [
            "main_cat",
            "sub_cat",
            "sub_sub_cat",
            "brand_name",
            "shipping",
        ]
        existing_categorical = [
            col for col in categorical_cols if col in self.train_vectorized.columns
        ]

        self.setup_result = setup(
            data=self.train_vectorized.assign(
                price=self.train["price"].reset_index(drop=True)
            ),
            target="price",
            session_id=session_id,
            categorical_features=existing_categorical if existing_categorical else None,
            normalize=True,
            transformation=True,
            verbose=True,
        )
        print("✅ PyCaret setup 완료")

    # ------------------ Base model 탐색 ------------------

    def find_base_model(self, sort_metric="R2"):
        if self.setup_result is None:
            raise ValueError("먼저 setup_pycaret()를 실행하세요.")

        print("🔍 Base model 탐색 시작...")
        self.best_model = compare_models(sort=sort_metric, n_select=1)
        print(f"🏆 Best model 선택 완료: {self.best_model}")
        return self.best_model

    # ------------------ 모델 성능 저장 ------------------
    def save_metrics(self, metrics_dict=None, model_name=None):
        if metrics_dict is None:
            if self.best_model is None:
                raise ValueError("모델이 없습니다.")
            pred = predict_model(self.best_model, data=self.train_vectorized)
            metrics_dict = {
                "R2": round(pred["R2"].iloc[0], 4) if "R2" in pred.columns else None,
                "RMSE": (
                    round(pred["RMSE"].iloc[0], 4) if "RMSE" in pred.columns else None
                ),
                "MAE": round(pred["MAE"].iloc[0], 4) if "MAE" in pred.columns else None,
            }
        self.metrics = metrics_dict

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        if model_name is None:
            model_name = str(self.best_model).split("(")[0]
        file_path = os.path.join(
            self.results_dir, f"{model_name}_metrics_{timestamp}.json"
        )

        with open(file_path, "w") as f:
            json.dump(self.metrics, f, indent=4)

        print(f"💾 Metrics 저장 완료: {file_path}")

    # ------------------ 시각화 ------------------
    def visualize_model(self, plots=["residuals", "feature"]):
        if self.best_model is None:
            raise ValueError("먼저 find_base_model()로 모델을 선택하세요.")

        print("🎨 시각화 시작...")
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        model_name = str(self.best_model).split("(")[0]

        for p in plots:
            try:
                save_path = os.path.join(
                    self.images_dir, f"{model_name}_{p}_{timestamp}.png"
                )
                plot_model(self.best_model, plot=p, save=True)
                print(f"✅ {p} plot 저장 완료: {save_path}")
            except Exception as e:
                print(f"⚠️ Plot {p} 실패: {e}")

    # ------------------ Test 예측 & submission ------------------
    def predict_test(self, submission_file="submission.csv"):
        if self.best_model is None:
            raise ValueError("먼저 find_base_model()로 모델을 선택하세요.")

        print("📦 Test 데이터 예측 시작...")
        predictions = predict_model(self.best_model, data=self.test_vectorized)

        submission = pd.DataFrame(
            {"test_id": self.test["test_id"], "price": predictions["Label"]}
        )

        submission_path = os.path.join(self.results_dir, submission_file)
        submission.to_csv(submission_path, index=False)
        print(f"💾 Submission 저장 완료: {submission_path}")
        return submission

In [2]:
# analyzer = MercariPyCaretAnalyzer()
# analyzer.load_data(train_file='train.tsv', test_file='test.tsv')
# analyzer.vectorize_text(method='tfidf', max_features=50000, n_components=100)
# analyzer.setup_pycaret()
# analyzer.find_base_model(sort_metric='R2')
# analyzer.save_metrics()
# analyzer.visualize_model(plots=['residuals','feature'])
# analyzer.predict_test(submission_file='submission.csv')

In [3]:
analyzer = MercariPyCaretAnalyzer()

In [4]:
analyzer.load_data()

📂 데이터 로딩 시작...
original data shape : train (1482535, 8), test (693359, 7)
✅ Price 외 결측치 처리 및 데이터 전처리 시작...
Price NaN count: 0
Final Price NaN count: 0
Train length: 1481661

Train head:
   train_id                                 name  item_condition_id  \
0         0  MLB Cincinnati Reds T Shirt Size XL                  3   
1         1     Razer BlackWidow Chroma Keyboard                  3   
2         2                       AVA-VIV Blouse                  1   
3         3                Leather Horse Statues                  1   
4         4                 24K GOLD plated rose                  1   

                                       category_name brand_name     price  \
0                                  Men/Tops/T-shirts    Unknown  2.397895   
1  Electronics/Computers & Tablets/Components & P...      Razer  3.970292   
2                        Women/Tops & Blouses/Blouse     Target  2.397895   
3                 Home/Home Décor/Home Décor Accents    Unknown  3.583519   
4 

In [5]:
analyzer.apply_undersampling(method="stratified", target_size=100000, n_bins=10)


🎯 Undersampling 시작...
   방법: stratified
   원본 크기: 1,481,661
   목표 크기: 100,000
   샘플링 비율: 6.75%
✅ Undersampling 완료
   최종 크기: 740,828
   실제 샘플링 비율: 50.00%
   제거된 샘플: 740,833

📊 샘플링 후 가격 분포:
   평균: 2.9809
   중앙값: 2.8904
   표준편차: 0.7462
   최소: 1.3863
   최대: 7.6034


In [6]:
analyzer.vectorize_text(method="tfidf", max_features=50000, n_components=100)

📝 텍스트 벡터화 및 차원 축소 시작...


Text columns:   0%|          | 0/2 [00:00<?, ?it/s]

▶ 컬럼: name


Text columns:  50%|█████     | 1/2 [00:47<00:47, 47.82s/it]

   ▪ 차원 축소 완료: 100 components
▶ 컬럼: item_description


Text columns:  50%|█████     | 1/2 [01:36<01:36, 96.79s/it]


KeyboardInterrupt: 

In [ ]:
analyzer.setup_pycaret()

🔧 PyCaret setup 시작...


,Description,Value
0,Session id,23
1,Target,price
2,Target type,Regression
3,Original data shape,"(1481661, 206)"
4,Transformed data shape,"(1481661, 216)"
5,Transformed train set shape,"(1037162, 216)"
6,Transformed test set shape,"(444499, 216)"
7,Numeric features,200
8,Categorical features,5
9,Preprocess,True


✅ PyCaret setup 완료


In [ ]:
analyzer.find_base_model(sort_metric="R2")

🔍 Base model 탐색 시작...


,,
,,
Initiated,. . . . . . . . . . . . . . . . . .,16:45:41
Status,. . . . . . . . . . . . . . . . . .,Fitting 10 Folds
Estimator,. . . . . . . . . . . . . . . . . .,Random Forest Regressor


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
knn,K Neighbors Regressor,0.4649,0.3911,0.6254,0.2978,0.1543,0.1621,320.9620
lr,Linear Regression,0.4910,0.4066,0.6377,0.2700,0.1574,0.1743,100.2130
ridge,Ridge Regression,0.4910,0.4066,0.6377,0.2700,0.1574,0.1743,82.7880
lar,Least Angle Regression,0.4910,0.4066,0.6377,0.2700,0.1574,0.1743,82.0380
br,Bayesian Ridge,0.4910,0.4066,0.6377,0.2700,0.1574,0.1743,102.0030
huber,Huber Regressor,0.4870,0.4115,0.6415,0.2613,0.1568,0.1695,95.1900
omp,Orthogonal Matching Pursuit,0.5251,0.4581,0.6769,0.1775,0.1667,0.1865,84.5800
lasso,Lasso Regression,0.5819,0.5570,0.7463,-0.0000,0.1850,0.2098,84.3690
en,Elastic Net,0.5819,0.5570,0.7463,-0.0000,0.1850,0.2098,84.5200
llar,Lasso Least Angle Regression,0.5819,0.5570,0.7463,-0.0000,0.1850,0.2098,83.2310


Processing:   0%|          | 0/85 [00:00<?, ?it/s]

In [ ]:
analyzer.save_metrics()

In [ ]:
analyzer.visualize_model(plots=["residuals", "feature"])

In [ ]:
analyzer.predict_test(submission_file="submission.csv")